In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## 1. DATA INGESTION & STRUCTURE

In [2]:
# Load dataset (Requirement: Load daily time-series data 2023–2025)
df = pd.read_csv('HHS_Unaccompanied_Alien_Children_Program.csv')

print("Initial Shape:", df.shape)
print("Raw Columns:", df.columns.tolist())
# Clean column names
df.columns = df.columns.str.strip()

# Standardize column names (Requirement: Dataset structuring)
df = df.rename(columns={
    df.columns[0]: 'Date',
    df.columns[1]: 'Apprehended',
    df.columns[2]: 'CBP_Custody',
    df.columns[3]: 'Transfers',
    df.columns[4]: 'HHS_Care',
    df.columns[5]: 'Discharges'
})

# Convert numeric columns
numeric_cols = ['Apprehended','CBP_Custody','Transfers','HHS_Care','Discharges']
for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(',', '', regex=False).astype(float)

Initial Shape: (1170, 6)
Raw Columns: ['Date', 'Children apprehended and placed in CBP custody*', 'Children in CBP custody', 'Children transferred out of CBP custody', 'Children in HHS Care', 'Children discharged from HHS Care']


In [3]:
# Shape of Dataset having only filled rows
print("Shape of Dataset with filled rows (Original dataset): ", df.dropna().shape)

Shape of Dataset with filled rows (Original dataset):  (720, 6)


In [4]:
# Convert Date to datetime (Requirement)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])

# Chronological ordering (Requirement)
df = df.sort_values('Date')

# Duplicate date check (Requirement: identify duplicated dates)
duplicate_count = df['Date'].duplicated().sum()
print("Duplicate Dates Found:", duplicate_count)

# Complete daily index creation (Requirement)
original_rows = len(df)
full_index = pd.date_range(df['Date'].min(), df['Date'].max(), freq='D')

df = df.set_index('Date').reindex(full_index)
df.index.name = 'Date'

# Fill stock variables forward
df[['CBP_Custody','HHS_Care']] = df[['CBP_Custody','HHS_Care']].ffill()

# Flow variables default to 0 if missing
df[['Apprehended','Transfers','Discharges']] = \
    df[['Apprehended','Transfers','Discharges']].fillna(0)

df = df.reset_index()

print("Original reported days:", original_rows)
print("Total calendar days:", len(df))
print("Date Range:", df['Date'].min().date(), "to", df['Date'].max().date())

Duplicate Dates Found: 0
Original reported days: 720
Total calendar days: 1075
Date Range: 2023-01-12 to 2025-12-21


In [5]:
df

,Date,Apprehended,CBP_Custody,Transfers,HHS_Care,Discharges
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0
1,2023-01-13,0.0,53.0,0.0,6566.0,0.0
2,2023-01-14,0.0,53.0,0.0,6566.0,0.0
3,2023-01-15,0.0,53.0,0.0,6566.0,0.0
4,2023-01-16,0.0,53.0,0.0,6566.0,0.0
...,...,...,...,...,...,...
1070,2025-12-17,7.0,31.0,11.0,2481.0,10.0
1071,2025-12-18,11.0,50.0,6.0,2472.0,16.0
1072,2025-12-19,0.0,50.0,0.0,2472.0,0.0
1073,2025-12-20,0.0,50.0,0.0,2472.0,0.0


## 2. DATA QUALITY & VALIDATION

In [6]:
# Identify Duplicate Dates (Requirement)
duplicate_dates = df['Date'].duplicated().sum()
print("Duplicate Dates Found:", duplicate_dates)

# Identify Missing Dates (Requirement)
# Since full daily index was created in Section 1,
# any missing original reporting days were already filled.
# Here we compare calendar span with original reporting count.

expected_days = (df['Date'].max() - df['Date'].min()).days + 1
actual_days = len(df)
print("Total Calendar Days:", expected_days)
print("Total Rows After Reindex:", actual_days)
print("Missing Dates Identified:", expected_days - actual_days)

# Validate Logical Constraints (Requirement)
# Transfers ≤ CBP custody
# Discharges ≤ HHS care

df['Transfers_Valid'] = df['Transfers'] <= df['CBP_Custody']
df['Discharges_Valid'] = df['Discharges'] <= df['HHS_Care']

print("Transfers Valid %:", round(df['Transfers_Valid'].mean()*100,2))
print("Discharges Valid %:", round(df['Discharges_Valid'].mean()*100,2))

# Flag Reporting Anomalies (Requirement)
df['Anomaly_Flag'] = ~(df['Transfers_Valid'] & df['Discharges_Valid'])

print("Total Anomalies Flagged:", df['Anomaly_Flag'].sum())
df.shape

Duplicate Dates Found: 0
Total Calendar Days: 1075
Total Rows After Reindex: 1075
Missing Dates Identified: 0
Transfers Valid %: 92.0
Discharges Valid %: 100.0
Total Anomalies Flagged: 86


(1075, 9)

In [7]:
df

,Date,Apprehended,CBP_Custody,Transfers,HHS_Care,Discharges,Transfers_Valid,Discharges_Valid,Anomaly_Flag
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,True,True,False
1,2023-01-13,0.0,53.0,0.0,6566.0,0.0,True,True,False
2,2023-01-14,0.0,53.0,0.0,6566.0,0.0,True,True,False
3,2023-01-15,0.0,53.0,0.0,6566.0,0.0,True,True,False
4,2023-01-16,0.0,53.0,0.0,6566.0,0.0,True,True,False
...,...,...,...,...,...,...,...,...,...
1070,2025-12-17,7.0,31.0,11.0,2481.0,10.0,True,True,False
1071,2025-12-18,11.0,50.0,6.0,2472.0,16.0,True,True,False
1072,2025-12-19,0.0,50.0,0.0,2472.0,0.0,True,True,False
1073,2025-12-20,0.0,50.0,0.0,2472.0,0.0,True,True,False


In [8]:
# Rows with Anomaly_Flag as "TRUE"
df[df['Anomaly_Flag']== True]

,Date,Apprehended,CBP_Custody,Transfers,HHS_Care,Discharges,Transfers_Valid,Discharges_Valid,Anomaly_Flag
12,2023-01-24,47.0,42.0,47.0,7433.0,175.0,False,True,True
13,2023-01-25,20.0,22.0,41.0,7538.0,180.0,False,True,True
21,2023-02-02,15.0,13.0,23.0,7879.0,298.0,False,True,True
41,2023-02-22,107.0,215.0,230.0,7978.0,232.0,False,True,True
42,2023-02-23,101.0,162.0,178.0,7914.0,386.0,False,True,True
...,...,...,...,...,...,...,...,...,...
749,2025-01-30,47.0,42.0,47.0,3923.0,159.0,False,True,True
752,2025-02-02,20.0,22.0,41.0,3483.0,168.0,False,True,True
759,2025-02-09,15.0,13.0,23.0,2878.0,99.0,False,True,True
763,2025-02-13,15.0,10.0,23.0,2703.0,72.0,False,True,True


## 3. DERIVED HEALTHCARE CAPACITY METRICS

In [9]:
# Total System Load (Requirement)
df['Total_System_Load'] = df['CBP_Custody'] + df['HHS_Care']

# Net Daily Intake (Requirement)
df['Net_Daily_Intake'] = df['Transfers'] - df['Discharges']

# Care Load Growth Rate (Requirement)
df['Care_Growth_Rate'] = df['Total_System_Load'].pct_change() * 100

# Interpretation of daily balance
positive_days = (df['Net_Daily_Intake'] > 0).sum()
negative_days = (df['Net_Daily_Intake'] < 0).sum()
balanced_days = (df['Net_Daily_Intake'] == 0).sum()

print("Days with Inflow > Outflow (Expansion Pressure):", positive_days)
print("Days with Outflow > Inflow (Relief Periods):", negative_days)
print("Balanced Days:", balanced_days)

# Backlog Indicator (Requirement)
df['Net_Intake_7Day'] = df['Net_Daily_Intake'].rolling(7, min_periods=1).mean()
df['Backlog_Indicator'] = (df['Net_Intake_7Day'] > 0).astype(int)

# Cumulative Backlog (Structural pressure analysis)
df['Cumulative_Net_Intake'] = df['Net_Daily_Intake'].cumsum()

# CBP vs HHS imbalance ratio (Pipeline balance analysis)
df['CBP_to_HHS_Ratio'] = df['CBP_Custody'] / df['HHS_Care']

print("Peak Total Load:", df['Total_System_Load'].max())
print("Average Net Intake:", df['Net_Daily_Intake'].mean())

df

Days with Inflow > Outflow (Expansion Pressure): 238
Days with Outflow > Inflow (Relief Periods): 475
Balanced Days: 362
Peak Total Load: 11762.0
Average Net Intake: -29.964651162790698


,Date,Apprehended,CBP_Custody,Transfers,HHS_Care,Discharges,Transfers_Valid,Discharges_Valid,Anomaly_Flag,Total_System_Load,Net_Daily_Intake,Care_Growth_Rate,Net_Intake_7Day,Backlog_Indicator,Cumulative_Net_Intake,CBP_to_HHS_Ratio
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,True,True,False,6619.0,-402.0,NaN,-402.000000,0,-402.0,0.008072
1,2023-01-13,0.0,53.0,0.0,6566.0,0.0,True,True,False,6619.0,0.0,0.000000,-201.000000,0,-402.0,0.008072
2,2023-01-14,0.0,53.0,0.0,6566.0,0.0,True,True,False,6619.0,0.0,0.000000,-134.000000,0,-402.0,0.008072
3,2023-01-15,0.0,53.0,0.0,6566.0,0.0,True,True,False,6619.0,0.0,0.000000,-100.500000,0,-402.0,0.008072
4,2023-01-16,0.0,53.0,0.0,6566.0,0.0,True,True,False,6619.0,0.0,0.000000,-80.400000,0,-402.0,0.008072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1070,2025-12-17,7.0,31.0,11.0,2481.0,10.0,True,True,False,2512.0,1.0,-0.396511,0.571429,1,-32199.0,0.012495
1071,2025-12-18,11.0,50.0,6.0,2472.0,16.0,True,True,False,2522.0,-10.0,0.398089,-0.714286,0,-32209.0,0.020227
1072,2025-12-19,0.0,50.0,0.0,2472.0,0.0,True,True,False,2522.0,0.0,0.000000,-0.714286,0,-32209.0,0.020227
1073,2025-12-20,0.0,50.0,0.0,2472.0,0.0,True,True,False,2522.0,0.0,0.000000,-0.714286,0,-32209.0,0.020227


## 4. TREND & TEMPORAL ANALYSIS

In [10]:
# DAILY, WEEKLY, MONTHLY TRENDS
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month_name()
df['Week'] = df['Date'].dt.to_period('W').astype(str)
df['Month_Year'] = df['Date'].dt.to_period('M').astype(str)

# Daily trends
print("DAILY TRENDS:")
print(f"   HHS Care - Peak: {df['HHS_Care'].max():,.0f}")
print(f"   HHS Care - Avg load in recent 30d: {df['HHS_Care'].tail(30).mean():,.0f}")

# Weekly trends
weekly = df.groupby('Week')['HHS_Care'].agg(['mean', 'max']).round(0)
print("\nWEEKLY TRENDS:")
print(f"   Busiest week avg: {weekly['mean'].max():,.0f}")
print(f"   Busiest week peak: {weekly['max'].max():,.0f}")

# Monthly trends  
monthly = df.groupby('Month_Year')['HHS_Care'].agg(['mean', 'max', 'count']).round(0)
print("\nMONTHLY TRENDS:")
print(f"   Busiest month: {monthly['mean'].idxmax()}")
print(f"   Busiest month peak: {monthly['max'].max():,.0f}")
print(f"   Busiest month avg: {monthly['mean'].max():,.0f}")

# SUSTAINED HIGH-LOAD PERIODS
print("\nSUSTAINED HIGH-LOAD PERIODS:")
hhs_30d_avg = df['HHS_Care'].rolling(30, min_periods=1).mean()
high_load_threshold = hhs_30d_avg.quantile(0.85)
sustained_high = (hhs_30d_avg > high_load_threshold).astype(int)
consecutive_high = sustained_high.groupby((sustained_high.diff() != 0).cumsum()).cumsum()
max_streak = consecutive_high.max()
print(f"   30-day high-load threshold: {high_load_threshold:,.0f}")
print(f"   Longest sustained period: {max_streak} consecutive days")
print(f"   High-load periods total: {(sustained_high == 1).sum()} days")

# EARLY vs LATE TIMELINE COMPARISON
midpoint = df['Date'].median()
early_period = df[df['Date'] < midpoint]['HHS_Care']
late_period = df[df['Date'] >= midpoint]['HHS_Care']

print("\nEARLY vs LATE TIMELINE:")
print(f"   Early period avg: {early_period.mean():,.0f}")
print(f"   Late period avg: {late_period.mean():,.0f}")
print(f"   Change: {((late_period.mean() - early_period.mean()) / early_period.mean() * 100):.0f}%")

DAILY TRENDS:
   HHS Care - Peak: 11,516
   HHS Care - Avg load in recent 30d: 2,425

WEEKLY TRENDS:
   Busiest week avg: 11,351
   Busiest week peak: 11,516

MONTHLY TRENDS:
   Busiest month: 2023-12
   Busiest month peak: 11,516
   Busiest month avg: 11,080

SUSTAINED HIGH-LOAD PERIODS:
   30-day high-load threshold: 8,704
   Longest sustained period: 162 consecutive days
   High-load periods total: 162 days

EARLY vs LATE TIMELINE:
   Early period avg: 8,390
   Late period avg: 3,777
   Change: -55%


##  5. PRESSURE & STRESS IDENTIFICATION

In [11]:
# Rolling averages to smooth daily fluctuations
df['HHS_7Day_Average'] = df['HHS_Care'].rolling(window=7).mean()
df['HHS_14Day_Average'] = df['HHS_Care'].rolling(window=14).mean()

# Volatility - how much daily numbers fluctuate over 7 days
df['Daily_Volatility'] = df['HHS_Care'].rolling(window=7).std()

# Stress threshold = top 20% busiest periods (80th percentile)
stress_threshold_7day = df['HHS_7Day_Average'].quantile(0.80)
stress_threshold_14day = df['HHS_14Day_Average'].quantile(0.80)

# Flag days where 7-day average exceeds stress threshold
df['Is_Stressed_7Day'] = df['HHS_7Day_Average'] > stress_threshold_7day

# Count consecutive stress days
df['Consecutive_Stress_Days'] = df['Is_Stressed_7Day'].groupby(
    (df['Is_Stressed_7Day'] != df['Is_Stressed_7Day'].shift()).cumsum()
).cumsum()

# Flag periods of prolonged strain (7+ consecutive stress days)
df['Prolonged_Strain'] = df['Consecutive_Stress_Days'] >= 7

# Results
print(f"7-day stress threshold: {stress_threshold_7day:,.0f} children")
print(f"14-day stress threshold: {stress_threshold_14day:,.0f} children")
print(f"Total prolonged strain periods: {df['Prolonged_Strain'].sum()}")
print(f"Days under stress (7-day avg): {df['Is_Stressed_7Day'].sum()}")


7-day stress threshold: 8,393 children
14-day stress threshold: 8,388 children
Total prolonged strain periods: 196
Days under stress (7-day avg): 214


In [12]:
df[df['Prolonged_Strain']==1]

,Date,Apprehended,CBP_Custody,Transfers,HHS_Care,Discharges,Transfers_Valid,Discharges_Valid,Anomaly_Flag,Total_System_Load,...,Year,Month,Week,Month_Year,HHS_7Day_Average,HHS_14Day_Average,Daily_Volatility,Is_Stressed_7Day,Consecutive_Stress_Days,Prolonged_Strain
118,2023-05-10,168.0,285.0,210.0,8681.0,304.0,True,True,False,8966.0,...,2023,May,2023-05-08/2023-05-14,2023-05,8676.000000,8529.357143,120.961426,True,7,True
119,2023-05-11,269.0,412.0,204.0,8672.0,392.0,True,True,False,9084.0,...,2023,May,2023-05-08/2023-05-14,2023-05,8659.142857,8566.000000,110.167189,True,8,True
120,2023-05-12,0.0,412.0,0.0,8672.0,0.0,True,True,False,9084.0,...,2023,May,2023-05-08/2023-05-14,2023-05,8642.285714,8602.642857,94.757083,True,9,True
121,2023-05-13,0.0,412.0,0.0,8672.0,0.0,True,True,False,9084.0,...,2023,May,2023-05-08/2023-05-14,2023-05,8625.428571,8639.285714,71.818886,True,10,True
122,2023-05-14,100.0,255.0,236.0,8445.0,288.0,True,True,False,8700.0,...,2023,May,2023-05-08/2023-05-14,2023-05,8619.714286,8645.928571,85.199206,True,11,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
430,2024-03-17,127.0,260.0,259.0,8373.0,342.0,True,True,False,8633.0,...,2024,March,2024-03-11/2024-03-17,2024-03,8696.714286,8759.071429,155.342326,True,29,True
431,2024-03-18,123.0,243.0,188.0,8333.0,283.0,True,True,False,8576.0,...,2024,March,2024-03-18/2024-03-24,2024-03,8636.142857,8735.714286,203.207143,True,30,True
432,2024-03-19,99.0,202.0,256.0,8390.0,197.0,False,True,True,8592.0,...,2024,March,2024-03-18/2024-03-24,2024-03,8571.428571,8706.428571,198.427029,True,31,True
433,2024-03-20,91.0,181.0,174.0,8365.0,227.0,True,True,False,8546.0,...,2024,March,2024-03-18/2024-03-24,2024-03,8506.571429,8670.357143,177.066708,True,32,True


## 6. KEY PERFORMANCE INDICATORS (ALL 5 REQUIRED)

In [13]:
print("\n KPIs:")

# 1. Peak children in care across entire period
peak_children = df['HHS_Care'].max()
print(f"   1. Peak children in care: {peak_children:,.0f}")

# 2. Average net intake pressure (new arrivals minus discharges, 7-day smooth)
net_intake_pressure = (df['CBP_Custody'] - df['Discharges']).rolling(7).mean()
avg_pressure = net_intake_pressure.mean()
print(f"   2. Net intake pressure (7d avg): {avg_pressure:,.0f}")

# 3. Care load stability (7-day volatility average)
daily_volatility = df['HHS_Care'].rolling(7).std()
volatility_index = daily_volatility.mean()
print(f"   3. Care load volatility: {volatility_index:,.0f}")

# 4. Weekly backlog accumulation (net weekly intake)
weekly_net_intake = (df['CBP_Custody'] - df['Discharges']).rolling(7).sum()
backlog_rate = weekly_net_intake.mean()
print(f"   4. Weekly backlog rate: {backlog_rate:,.0f} children/week")

# 5. Discharge effectiveness (discharges relative to prior day load)
discharge_ratio = df['Discharges'] / df['HHS_Care'].shift(1)
avg_discharge_ratio = discharge_ratio.mean()
print(f"   5. Discharge effectiveness: {avg_discharge_ratio:.1%}")


 KPIs:
   1. Peak children in care: 11,516
   2. Net intake pressure (7d avg): 54
   3. Care load volatility: 102
   4. Weekly backlog rate: 376 children/week
   5. Discharge effectiveness: 1.6%


In [14]:
print("\n" + "="*50)
print("UAC PROGRAM CAPACITY ANALYSIS - EXECUTIVE SUMMARY")
print("="*50)
print(f"Dataset: {len(df)} days | {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Peak Crisis: {df['HHS_Care'].max():,.0f} children in HHS care")
print(f"System Peak: {df['Total_System_Load'].max():,.0f} total responsibility") 
print(f"Prolonged strain: {df['Prolonged_Strain'].sum()} periods (7+ consecutive days)")
print(f"Recent load: ~{df['HHS_Care'].tail(30).mean():,.0f} children")


UAC PROGRAM CAPACITY ANALYSIS - EXECUTIVE SUMMARY
Dataset: 1075 days | 2023-01-12 to 2025-12-21
Peak Crisis: 11,516 children in HHS care
System Peak: 11,762 total responsibility
Prolonged strain: 196 periods (7+ consecutive days)
Recent load: ~2,425 children


In [15]:
print("\n" + "="*50)
print("EXECUTIVE INSIGHTS SUMMARY")
print("="*50)

print("\n1. TOTAL CARE SYSTEM LOAD")
print(f"   Peak: {df['HHS_Care'].max():,.0f}")
print(f"   Average: {df['HHS_Care'].mean():,.0f}/day")
print(f"   Recent 30-day: {df['HHS_Care'].tail(30).mean():,.0f}")

print("\n2. INFLOW vs OUTFLOW BALANCE")
net_intake = df['CBP_Custody'] - df['Discharges']
print(f"   Net Intake Pressure: {net_intake.mean():+.0f}/day")
print(f"   Transfers avg: {df['Transfers'].mean():,.0f}/day")
print(f"   Discharges avg: {df['Discharges'].mean():,.0f}/day")

print("\n3. CAPACITY STRESS PERIODS")
print(f"   HHS Care Peak: {df['HHS_Care'].max():,.0f}")
print(f"   Stress Days: {df['Is_Stressed_7Day'].sum()}")
print(f"   Prolonged Strain: {df['Prolonged_Strain'].sum()} periods")

print("\n4. OPERATIONAL EFFICIENCY")
discharge_ratio = df['Discharges'] / df['HHS_Care'].shift(1)
print(f"   Discharge Ratio: {discharge_ratio.mean():.1%}")
volatility_7d = df['HHS_Care'].rolling(7).std()
print(f"   Volatility Index: {volatility_7d.mean():.0f}")

print("\n5. DATA COVERAGE")
print(f"   Records processed: {len(df)} days")
print(f"   Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")


EXECUTIVE INSIGHTS SUMMARY

1. TOTAL CARE SYSTEM LOAD
   Peak: 11,516
   Average: 6,081/day
   Recent 30-day: 2,425

2. INFLOW vs OUTFLOW BALANCE
   Net Intake Pressure: +53/day
   Transfers avg: 86/day
   Discharges avg: 116/day

3. CAPACITY STRESS PERIODS
   HHS Care Peak: 11,516
   Stress Days: 214
   Prolonged Strain: 196 periods

4. OPERATIONAL EFFICIENCY
   Discharge Ratio: 1.6%
   Volatility Index: 102

5. DATA COVERAGE
   Records processed: 1075 days
   Date range: 2023-01-12 to 2025-12-21


In [16]:
df.shape

(1075, 26)